# Gold Google Trends Daily

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns

In [3]:
gold_google_trends_df = pd.read_csv('../raw/gold_google_trends_daily.csv', sep=',', on_bad_lines='skip')
gold_google_trends_df.info()
gold_google_trends_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1351 entries, 0 to 1350
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          1351 non-null   object 
 1   ohl_interest  1351 non-null   float64
 2   match_id      142 non-null    object 
dtypes: float64(1), object(2)
memory usage: 31.8+ KB


,date,ohl_interest,match_id
0,2022-07-01,2.43,NaN
1,2022-07-02,4.25,NaN
2,2022-07-03,4.56,NaN
3,2022-07-04,3.04,NaN
4,2022-07-05,3.65,NaN


## Converting the rows to the correct Data Types

Parse "```date```" to datetime

In [4]:
gold_google_trends_df['date'] = pd.to_datetime(gold_google_trends_df['date'], errors='coerce')

## Dealing with NULL and duplicate values

**Duplicates**

In [5]:
print("Duplicate rows:", gold_google_trends_df.duplicated().sum())
print("Duplicate dates:", gold_google_trends_df['date'].duplicated().sum())

Duplicate rows: 0
Duplicate dates: 1


In [8]:
# Find which date is duplicated
dup_date = gold_google_trends_df[gold_google_trends_df['date'].duplicated(keep=False)]
print(dup_date)

           date  ohl_interest match_id
1307 2026-01-28          5.91      NaN
1308 2026-01-28          6.00      NaN


2026-01-28 appeared twice with no match_id and near-identical interest values (5.91 and 6.00). Rows were merged by averaging ohl_interest.

In [9]:
# Average the duplicate date, then drop the duplicate
gold_google_trends_df = (
    gold_google_trends_df
    .groupby('date', as_index=False)
    .agg({'ohl_interest': 'mean', 'match_id': 'first'})
    .sort_values('date')
    .reset_index(drop=True)
)

print("Rows after dedup:", len(gold_google_trends_df))  # should be 1350

Rows after dedup: 1350


**Date sanity check**

In [6]:
print("Date range:", gold_google_trends_df['date'].min(), "→", gold_google_trends_df['date'].max())
print("Coerced NaT count:", gold_google_trends_df['date'].isna().sum())

Date range: 2022-07-01 00:00:00 → 2026-03-11 00:00:00
Coerced NaT count: 0


**Interest score range**

In [7]:
print("ohl_interest stats:")
print(gold_google_trends_df['ohl_interest'].describe())

ohl_interest stats:
count    1351.000000
mean        6.095470
std         9.887924
min         0.000000
25%         2.350000
50%         3.480000
75%         5.135000
max       103.000000
Name: ohl_interest, dtype: float64


Google Trends Scores are tipically between 0 and 100, but in our dataset, the max is 103.

In [10]:
# Checking which rows are over 100
print(gold_google_trends_df[gold_google_trends_df['ohl_interest'] > 100])

           date  ohl_interest                   match_id
1331 2026-02-21         103.0  exzov7o3p9v5vqr86qipfm1w4


The OHL interest is linked to a match. So, before deciding what to do, we have to check what match it is.

In [11]:
gold_match_df = pd.read_csv('../raw/gold_match.csv', sep=',', on_bad_lines='skip')
print(gold_match_df[gold_match_df['match_id'] == 'exzov7o3p9v5vqr86qipfm1w4'][
    ['match_id', 'match_date', 'home_team', 'away_team', 'competition_name']
])

                      match_id  match_date    home_team  away_team  \
139  exzov7o3p9v5vqr86qipfm1w4  2026-02-21  Club Brugge  OH Leuven   

               competition_name  
139  Belgian Jupiler Pro League  


```ohl_interest``` of 103.0 on 2026-02-21 is linked to the OHL away match vs Club Brugge (Belgian Jupiler Pro League). Club Brugge is the highest-profile opponent in Belgian football, making this spike plausible.

In [12]:
gold_google_trends_df['ohl_interest'] = gold_google_trends_df['ohl_interest'].clip(upper=100)

# Verify
print(gold_google_trends_df['ohl_interest'].max())  # should be 100.0

100.0


```ohl_interest``` capped at 100. One value (2026-02-21, Club Brugge away) had a score of 103.0, likely due to a normalization artifact when stitching together overlapping API batches.

**Validate the "```match_id```" join key**

In [13]:
overlap = gold_google_trends_df['match_id'].dropna().isin(gold_match_df['match_id'])
print(f"Matched: {overlap.sum()} / {gold_google_trends_df['match_id'].dropna().shape[0]} ({overlap.mean():.1%})")

Matched: 142 / 142 (100.0%)


## Building the per-match aggregation

In [14]:
# Merge trends with match dates to compute rolling pre-match interest
trends_with_dates = gold_google_trends_df.dropna(subset=['match_id']).merge(
    gold_match_df[['match_id', 'match_date']],
    on='match_id',
    how='left'
)

# For each match, get avg ohl_interest in the 7 days before
# (requires a date-range join — simplest approach via merge + filter)
match_dates = gold_match_df[['match_id', 'match_date']].copy()
match_dates['match_date'] = pd.to_datetime(match_dates['match_date'])

trends_clean = gold_google_trends_df[['date', 'ohl_interest']].copy()

rows = []
for _, row in match_dates.iterrows():
    window = trends_clean[
        (trends_clean['date'] >= row['match_date'] - pd.Timedelta(days=7)) &
        (trends_clean['date'] < row['match_date'])
    ]
    rows.append({
        'match_id': row['match_id'],
        'avg_ohl_interest_7d': window['ohl_interest'].mean()
    })

trends_agg = pd.DataFrame(rows)
trends_agg.head()

,match_id,avg_ohl_interest_7d
0,d0te6swsv2y99ywgugc2utbmc,5.468571
1,d256yo3eng04m0fu7b4sl7wno,5.165714
2,d3pqkck2grzx98jg4sofhp8us,5.772857
3,d4mn5ksbxuvnaww4pmommxhqs,7.812857
4,d5htdqmc8w72upys41sfxhfkk,8.117143


In [15]:
trends_agg.to_csv('../cleaned/match_trends.csv', index=False)